In [ ]:
# 1. Force install and import gTTS in the same session
import sys
!{sys.executable} -m pip install -q gtts datasets transformers accelerate torch pandas scikit-learn

from gtts import gTTS
import IPython.display as ipd
from datasets import load_dataset
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

print("=" * 60)
print("INITIALIZING OMNICAST MULTILINGUAL ADVANCED AI PIPELINE")
print("=" * 60)

# 2. Multi-Dataset Loading
print("Loading datasets from Hugging Face...")
cricket_ds = load_dataset("VinitT/Cricket-Commentary", split="train")
df_cricket = cricket_ds.to_pandas()

gcb_ds = load_dataset("A4Blind/GCB", split="test")
df_gcb = gcb_ds.to_pandas()

qa_ds = load_dataset("PedroCJardim/QASports", "soccer", split="train", streaming=True)
qa_samples = list(qa_ds.take(50))
df_qa = pd.DataFrame(qa_samples)

print(f"✓ Datasets loaded successfully! Cricket: {len(df_cricket)} | GCB: {len(df_gcb)} | QASports: {len(df_qa)}")

# 3. Clean commentary lines
def clean_commentary(text):
    if pd.isna(text):
        return ""
    return str(text).replace("You are an expert cricket commentator.", "").strip()

df_cricket['cleaned_commentary'] = df_cricket['output'].apply(clean_commentary)
mvp_feed = df_cricket['cleaned_commentary'].dropna().head(10).reset_index(drop=True)

# 4. Load AI Model
model_id = "Qwen/Qwen2-0.5B-Instruct"
print(f"Loading model: {model_id}...")

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
)

commentary_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=128
)
print("AI Engine loaded successfully!")

# 5. Language Mapping for all 15 languages
LANGUAGES_CONFIG = {
    "English": {"name": "English", "gtts": "en"},
    "Hindi": {"name": "Hindi", "gtts": "hi"},
    "Marathi": {"name": "Marathi", "gtts": "mr"},
    "Punjabi": {"name": "Punjabi", "gtts": "pa"},
    "Odia": {"name": "Odia", "gtts": "or"},
    "Gujarati": {"name": "Gujarati", "gtts": "gu"},
    "Rajasthani": {"name": "Rajasthani", "gtts": "hi"},
    "Tamil": {"name": "Tamil", "gtts": "ta"},
    "Telugu": {"name": "Telugu", "gtts": "te"},
    "Nepali": {"name": "Nepali", "gtts": "ne"},
    "Bengali": {"name": "Bengali", "gtts": "bn"},
    "Kannada": {"name": "Kannada", "gtts": "kn"},
    "Urdu": {"name": "Urdu", "gtts": "ur"},
    "Assamese": {"name": "Assamese", "gtts": "as"},
    "Bhojpuri": {"name": "Bhojpuri", "gtts": "hi"}
}

def adapt_commentary_multilingual(raw_text, audience_mode="rookie", target_lang="Hindi"):
    lang_info = LANGUAGES_CONFIG.get(target_lang, {"name": "English", "gtts": "en"})
    lang_name = lang_info["name"]

    if audience_mode == "rookie":
        content = (
            f"Translate and rewrite this cricket commentary into {lang_name}. "
            f"Make it suitable for an absolute beginner who doesn't know cricket rules: '{raw_text}'"
        )
    else:
        content = (
            f"Translate and enhance this cricket commentary into {lang_name} with hardcore tactical analysis: '{raw_text}'"
        )

    messages = [{"role": "user", "content": content}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    response = commentary_pipeline(
        prompt, max_new_tokens=120, do_sample=True, temperature=0.7, pad_token_id=tokenizer.eos_token_id
    )
    generated_output = response[0]["generated_text"]
    return generated_output.split("<|im_start|>assistant\n")[-1].replace("<|im_end|>", "").strip()

# 6. Generate Tabular Output Across Languages
sample_ball = mvp_feed[0] if len(mvp_feed) > 0 else "Short of a length and angled across off, left alone by the batsman."
print(f"\nProcessing sample ball across 15 languages:\n'{sample_ball}'\n")

multilingual_results = []
for lang_key in LANGUAGES_CONFIG.keys():
    translated_text = adapt_commentary_multilingual(sample_ball, audience_mode="rookie", target_lang=lang_key)
    multilingual_results.append({
        "Language": lang_key,
        "gTTS Code": LANGUAGES_CONFIG[lang_key]["gtts"],
        "Adapted Commentary": translated_text
    })

df_multilingual = pd.DataFrame(multilingual_results)
display(df_multilingual)

# 7. Generate Multilingual Voice Mail
selected_lang = "Bengali"  # Change to any language from the table
row_data = df_multilingual[df_multilingual["Language"] == selected_lang].iloc[0]
text_to_speak = row_data["Adapted Commentary"]
tts_code = row_data["gTTS Code"]

print(f"\nGenerating voice mail audio for {selected_lang} ({tts_code})...")
tts = gTTS(text=text_to_speak, lang=tts_code)
audio_filename = f"omnicast_{selected_lang.lower()}_voice_mail.mp3"
tts.save(audio_filename)

print("Voice mail created successfully! Playing below:")
ipd.Audio(audio_filename, autoplay=False)

INITIALIZING OMNICAST MULTILINGUAL ADVANCED AI PIPELINE
Loading datasets from Hugging Face...
✓ Datasets loaded successfully! Cricket: 91679 | GCB: 20167 | QASports: 50
Loading model: Qwen/Qwen2-0.5B-Instruct...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

AI Engine loaded successfully!

Processing sample ball across 15 languages:
'short of a length and some shape into off stump. Allows it come on and dabs with soft hands in front of point'



,Language,gTTS Code,Adapted Commentary
0,English,en,"""Short, no length, and a shape that can be bro..."
1,Hindi,hi,"""कम बड़े और किसी प्रमुख चौथे हो। यहां का मैच ल..."
2,Marathi,mr,"""कैसे आपको क्या पहुंच सकते हैं? मैंने कहा कि उ..."
3,Punjabi,pa,"""लाईल लोग गर्दित रहे, एक पूजा और कुछ आकाश है। ..."
4,Odia,or,"""क्या नाही, कोरिडिंग कसे प्रतिसादने स्थापित के..."
5,Gujarati,gu,"""कैलो की छात्र अपने स्वर्ग में रहता है, और यह ..."
6,Rajasthani,hi,"The commentary in English would be:\n\n""Short ..."
7,Tamil,ta,"Translation and rewriting:\n\n""Short of a leng..."
8,Telugu,te,"""Short of a length, some shape into the off st..."
9,Nepali,ne,"""माझ्या लोकांनी किनारे आता गर्दैन, एक छात्रे ज..."



Generating voice mail audio for Bengali (bn)...
Voice mail created successfully! Playing below:
